# BSM L05E — Biometria jako *lokalna bramka* do sekretu (SecretBox + policy)

## Tryb pracy
To nie jest lab z Pythonem. Notebook jest tylko **formularzem odpowiedzi** i checklistą.
Kod piszesz w **Android Studio / Kotlin** w projekcie:

- `student/apps/lesson_e_app`

## Przygotowanie przed zajęciami (zrób zanim zaczniesz zadania)

### A. Import projektu w Android Studio (klik po kliku)
1. Uruchom Android Studio.
2. Wybierz: **File → Open...**
3. Wskaż folder projektu: `student/apps/lesson_e_app`
4. Jeśli Android Studio zapyta o zaufanie do projektu: wybierz **Trust Project**.
5. Poczekaj na zakończenie synchronizacji Gradle (na dole zobaczysz pasek postępu).

### B. Jeśli Gradle Sync nie startuje lub stoi
1. Kliknij: **File → Sync Project with Gradle Files**
2. Jeśli pojawi się informacja o wymaganej wersji JDK, wybierz JDK 17:
   - **File → Settings → Build, Execution, Deployment → Build Tools → Gradle**
   - **Gradle JDK → 17**

### C. Uruchom testy jednostkowe (bez emulatora)
W tym labie wszystko sprawdzamy przez testy JVM.

Opcja 1 (Android Studio):
1. Otwórz okno Gradle (zwykle po prawej).
2. Rozwiń: `lesson_e_app` → `app` → **Tasks** → **verification**
3. Uruchom: **testDebugUnitTest**

Jeśli ten krok nie działa, **nie idź dalej** — najpierw napraw środowisko.

## Cel labu
Wykład 5: biometria zwykle nie jest „sekretem do wysłania”, tylko mechanizmem lokalnym:
- biometria/credential **odblokowuje dostęp** do klucza / sekretu na urządzeniu,
- a dopiero potem aplikacja może wykonać wrażliwą operację.

W tym labie implementujesz:
1) `SecretBox` — minimalne szyfrowanie uwierzytelnione AES-GCM,
2) `BiometricBoundSecretStore` — politykę: sekret można ujawnić tylko z „świeżym” tokenem bramki.

Wszystko weryfikujesz przez testy w `app/src/test/...`.


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "
" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("\n", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    wyslij_odpowiedz(task_id, final_answer)


In [ ]:
#@title Konfiguracja ścieżki do projektu (wymagane do generowania dowodu)
# Wpisz ścieżkę do folderu projektu Android (dokładnie ten folder, który otwierasz w Android Studio).
# Domyślnie w repo: student/apps/lesson_e_app
PROJECT_ROOT = "student/apps/lesson_e_app"  #@param {type:"string"}

import os
import glob
import xml.etree.ElementTree as ET

TEST_RESULTS_GLOB = "app/build/test-results/testDebugUnitTest/TEST-*.xml"


def _load_junit_xml_files(project_root: str):
    pattern = os.path.join(project_root, TEST_RESULTS_GLOB)
    files = sorted(glob.glob(pattern))
    return files


def _parse_suite(xml_path: str):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    if root.tag != "testsuite":
        # Gradle usually emits <testsuite>, but be defensive.
        return None
    return root


def _suite_classname(suite):
    # Gradle sets suite name to fully-qualified class name.
    return suite.attrib.get("name", "")


def _suite_counts(suite):
    tests = int(suite.attrib.get("tests", "0"))
    failures = int(suite.attrib.get("failures", "0"))
    errors = int(suite.attrib.get("errors", "0"))
    skipped = int(suite.attrib.get("skipped", "0"))
    return tests, failures, errors, skipped


def _case_status(suite, test_method: str):
    for case in suite.findall("testcase"):
        if case.attrib.get("name") != test_method:
            continue
        # failure/error present => not ok
        if case.find("failure") is not None or case.find("error") is not None:
            return "FAIL"
        return "PASS"
    return "MISSING"


def evidence_for(project_root: str, fqcn: str, required_tests: list[str], task_id: str):
    xml_files = _load_junit_xml_files(project_root)
    if not xml_files:
        return f"{task_id}|MISSING_TEST_RESULTS"

    suite = None
    for path in xml_files:
        s = _parse_suite(path)
        if s is None:
            continue
        if _suite_classname(s) == fqcn:
            suite = s
            break

    if suite is None:
        return f"{task_id}|MISSING_SUITE={fqcn}"

    tests, failures, errors, skipped = _suite_counts(suite)
    statuses = []
    ok = True
    for name in required_tests:
        st = _case_status(suite, name)
        statuses.append(f"{name}={st}")
        ok = ok and (st == "PASS")

    ok = ok and failures == 0 and errors == 0
    ok_s = "YES" if ok else "NO"
    status_s = ",".join(statuses)
    return f"{task_id}|ok={ok_s}|tests={tests}|failures={failures}|errors={errors}|skipped={skipped}|{status_s}"


def evidence_overall(project_root: str):
    xml_files = _load_junit_xml_files(project_root)
    if not xml_files:
        return "E04|MISSING_TEST_RESULTS"

    total_tests = total_failures = total_errors = total_skipped = 0
    for path in xml_files:
        s = _parse_suite(path)
        if s is None:
            continue
        t,f,e,sk = _suite_counts(s)
        total_tests += t
        total_failures += f
        total_errors += e
        total_skipped += sk

    ok = (total_failures == 0 and total_errors == 0)
    ok_s = "YES" if ok else "NO"
    return f"E04|ok={ok_s}|tests={total_tests}|failures={total_failures}|errors={total_errors}|skipped={total_skipped}"


# E02 — SecretBox: AES-GCM (szyfrowanie uwierzytelnione)

## Co edytujesz
Plik: `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/SecretBox.kt`

W tym pliku masz trzy ważne elementy:
- `encrypt(plaintext, iv)` — masz użyć **podanego IV** i zwrócić bajty `iv || ciphertextAndTag`
- `decrypt(message)` — ma umieć rozpakować format `iv || ciphertextAndTag` i zwrócić `plaintext`
- `cipherEncrypt(...)` / `cipherDecrypt(...)` — helpery do inicjalizacji `Cipher`

## Dokładnie gdzie pisać kod
W `SecretBox.kt` uzupełnij:
- `TODO(L05-1)` w `encrypt(...)`
- `TODO(L05-2)` w `decrypt(...)`

## Wymagania (sprawdza je test)
- Algorytm: `AES/GCM/NoPadding`
- Tag: 128 bitów (już ustawione w `GCMParameterSpec`)
- IV ma mieć długość `SecretBox.IV_BYTES` (12 bajtów)
  - jeśli IV ma inną długość → rzuć `IllegalArgumentException`
- Zwracany format to **dokładnie**: `iv || ciphertextAndTag`
- `decrypt(...)` ma zwracać `null` jeśli:
  - wiadomość jest za krótka
  - autentykacja GCM nie przejdzie (ktoś zmienił bajty)

## Jak uruchomić testy tylko do tego zadania
Uruchom test:
- `app/src/test/java/com/example/secretlab/secure/SecretBoxStudentTest.kt`

W Android Studio:
1. Otwórz plik testu.
2. Kliknij zieloną strzałkę przy nazwie klasy albo testu.


In [ ]:
#@title E02 — Formularz odpowiedzi
secretbox_tests_passed = ""  #@param ["", "YES", "NO"]

# Dowód (wygenerowany automatycznie z wyników testów).
evidence = evidence_for(PROJECT_ROOT, 'com.example.secretlab.secure.SecretBoxStudentTest', ['encryptsAsIvPlusCiphertextAndDecryptsBack','returnsNullWhenMessageTooShort','detectsTamperingAndReturnsNull','rejectsWrongIvLengthInEncrypt'], 'E02')
print('EVIDENCE:', evidence)

final_answer = prepare_answer(
    "pass=" + secretbox_tests_passed,
    "evidence=" + short_text(evidence, limit=180),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E02", final_answer)


# E03 — BiometricBoundSecretStore: polityka „lokalnej bramki”

## Co edytujesz
Plik: `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/BiometricBoundSecretStore.kt`

## Dokładnie gdzie pisać kod
Uzupełnij `TODO(L05-3)` w metodzie:
- `revealSecret(token: GateToken?)`

## Polityka, którą masz wymusić
Sekret wolno ujawnić tylko wtedy, gdy:

Ważne: `clock()` zwraca **epoch seconds** (sekundy), a nie milisekundy. `GateToken.issuedAtEpochSeconds` jest w tych samych jednostkach.
1. `token` nie jest `null`
2. token jest wystarczająco „świeży”:
   - `age = clock() - token.issuedAtEpochSeconds`
   - wymaganie: `0 <= age <= maxTokenAgeSeconds`
   - jeśli `age < 0` (token „z przyszłości”) → traktuj jako niespełnione warunki i zwróć `null`

Jeśli warunki nie są spełnione → zwróć `null`.

## Jak uruchomić testy tylko do tego zadania
Uruchom test:
- `app/src/test/java/com/example/secretlab/secure/BiometricBoundSecretStoreStudentTest.kt`


In [ ]:
#@title E03 — Formularz odpowiedzi
store_tests_passed = ""  #@param ["", "YES", "NO"]

# Dowód (wygenerowany automatycznie z wyników testów).
evidence = evidence_for(PROJECT_ROOT, 'com.example.secretlab.secure.BiometricBoundSecretStoreStudentTest', ['refusesToRevealSecretWithoutToken','refusesToRevealSecretWhenTokenIsTooOld','revealsSecretWhenTokenIsFreshEnough','refusesToRevealSecretWhenTokenIsFromFuture'], 'E03')
print('EVIDENCE:', evidence)

final_answer = prepare_answer(
    "pass=" + store_tests_passed,
    "evidence=" + short_text(evidence, limit=180),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E03", final_answer)


# E04 — Checkpoint (uruchom cały pakiet testów)

Na koniec uruchom w Android Studio: **Gradle → Tasks → verification → testDebugUnitTest**.

Wpisz tylko, czy cały pakiet testów przeszedł.


In [ ]:
#@title E04 — Formularz odpowiedzi
all_tests_passed = ""  #@param ["", "YES", "NO"]

# Dowód (wygenerowany automatycznie z wyników testów).
evidence = evidence_overall(PROJECT_ROOT)
print('EVIDENCE:', evidence)

final_answer = prepare_answer(
    "pass=" + all_tests_passed,
    "evidence=" + short_text(evidence, limit=180),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E04", final_answer)


# Eksport odpowiedzi (co oddać)

Ten notebook **nie wysyła** odpowiedzi automatycznie.

1. Uzupełnij i uruchom komórki formularzy `E02`, `E03`, `E04`.
2. Następnie uruchom ostatnią komórkę **Export** poniżej.
3. Skopiuj wydrukowany blok i wklej do LMS / wyślij mailem — dokładnie tak, jak wymaga prowadzący.


Uwaga: każda odpowiedź zawiera pole `evidence=...` wygenerowane z plików JUnit XML.


In [ ]:
#@title Export — skopiuj ten blok jako odpowiedź końcową
print("=== BSM L05E ANSWERS ===")
for k in sorted(answers.keys()):
    print(f"{k}: {answers[k]}")
print("========================")
